# 05.03.01 XGBoost Classifier Benchmark

**Phase 05.03 — Gradient Boosting Models (Tabular Only)**  
*OFPHIL V2: Charity Patient Eligibility Classification*

---

### Description & Purpose
Train and benchmark an eXtreme Gradient Boosting (XGBoost) model on the 48 tabular features using Stratified 5-Fold Cross-Validation loaded from `data/processed/splits/cv_split_indices.json`. Evaluates learning rates, tree depth, and early stopping on fold validation partitions.

**Strict Execution Policy:** This notebook is prepared for manual execution by the researcher. No cell execution outputs are pre-populated.

### Research Objectives

1. Establish regularized gradient boosting performance on tabular charity patient data.
2. Optimize hyperparameters: learning rate ($\eta \in [0.05, 0.1]$), max tree depth ($max\_depth \in [3, 5, 7]$), and subsampling.
3. Compute out-of-fold metrics across 5 Stratified folds: Accuracy, Precision, Recall, Macro F1, and ROC-AUC.
4. Extract gain-based feature importances and log performance to `models/trained/gradient_boosting/xgboost_metrics.json`.

### Inputs & Expected Outputs

**Required Inputs:**
- `data/interim/features/selected_features.csv` (Primary tabular features)
- `data/processed/splits/cv_split_indices.json` (Stratified 5-Fold split definitions)

**Expected Outputs:**
- `models/trained/gradient_boosting/xgboost_metrics.json` (Cross-validation metrics and fold logs)
- `models/trained/gradient_boosting/xgboost_feature_importance.csv` (Gain-based feature importances)

### 1. Environment Setup & Library Imports

In [ ]:
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"XGBoost version: {xgb.__version__}, RANDOM_SEED={RANDOM_SEED}")


### 2. Setup Paths & Load Data and Fold Indices

In [ ]:
PROJECT_ROOT = Path("../../../..")
FEATURES_PATH = PROJECT_ROOT / "data" / "interim" / "features" / "selected_features.csv"
CV_PATH = PROJECT_ROOT / "data" / "processed" / "splits" / "cv_split_indices.json"
OUTPUT_DIR = PROJECT_ROOT / "models" / "trained" / "gradient_boosting"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert FEATURES_PATH.exists(), f"Missing features dataset: {FEATURES_PATH}"
assert CV_PATH.exists(), f"Missing CV split indices: {CV_PATH}"

df_features = pd.read_csv(FEATURES_PATH)
with open(CV_PATH, "r", encoding="utf-8") as f:
    cv_data = json.load(f)

feature_cols = cv_data["feature_columns"]
train_mask = df_features["split_partition"] == "train"
df_train = df_features.loc[train_mask].reset_index(drop=True)

X_train_full = df_train[feature_cols]
y_train_full = (df_train["eligibility"] == "ELIGIBLE").astype(int)

print(f"Loaded training development set: X={X_train_full.shape}, y={y_train_full.shape}")


### 3. Hyperparameter Tuning Across 5-Fold Stratified Cross-Validation

In [ ]:
param_candidates = [
    {"learning_rate": 0.05, "max_depth": 3, "n_estimators": 150, "subsample": 0.8},
    {"learning_rate": 0.05, "max_depth": 5, "n_estimators": 150, "subsample": 0.8},
    {"learning_rate": 0.10, "max_depth": 3, "n_estimators": 100, "subsample": 0.9},
    {"learning_rate": 0.10, "max_depth": 5, "n_estimators": 100, "subsample": 0.9}
]

grid_results = []
for params in param_candidates:
    fold_f1s = []
    for fold_info in cv_data["folds"]:
        t_idx = fold_info["train_indices"]
        v_idx = fold_info["val_indices"]
        
        X_tr, y_tr = X_train_full.iloc[t_idx], y_train_full.iloc[t_idx]
        X_va, y_va = X_train_full.iloc[v_idx], y_train_full.iloc[v_idx]
        
        clf = XGBClassifier(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            learning_rate=params["learning_rate"],
            subsample=params["subsample"],
            eval_metric="logloss",
            random_state=RANDOM_SEED,
            n_jobs=-1
        )
        clf.fit(X_tr, y_tr)
        preds = clf.predict(X_va)
        fold_f1s.append(f1_score(y_va, preds, average="macro"))
    
    mean_f1 = float(np.mean(fold_f1s))
    grid_results.append({"params": params, "mean_macro_f1": mean_f1})
    print(f"Params {params} -> Mean Macro F1: {mean_f1:.4f}")

best_cfg = max(grid_results, key=lambda x: x["mean_macro_f1"])
best_params = best_cfg["params"]
print(f"Optimal XGBoost params selected: {best_params}")


### 4. Cross-Validation Evaluation with Optimal Hyperparameters

In [ ]:
fold_evaluations = []
oof_predictions = np.zeros(len(y_train_full))
oof_probabilities = np.zeros(len(y_train_full))

start_time = time.time()
for fold_info in cv_data["folds"]:
    f_num = fold_info["fold"]
    t_idx = fold_info["train_indices"]
    v_idx = fold_info["val_indices"]
    
    X_tr, y_tr = X_train_full.iloc[t_idx], y_train_full.iloc[t_idx]
    X_va, y_va = X_train_full.iloc[v_idx], y_train_full.iloc[v_idx]
    
    xgb_model = XGBClassifier(
        n_estimators=best_params["n_estimators"],
        max_depth=best_params["max_depth"],
        learning_rate=best_params["learning_rate"],
        subsample=best_params["subsample"],
        eval_metric="logloss",
        random_state=RANDOM_SEED,
        n_jobs=-1
    )
    xgb_model.fit(X_tr, y_tr)
    val_preds = xgb_model.predict(X_va)
    val_probs = xgb_model.predict_proba(X_va)[:, 1]
    
    oof_predictions[v_idx] = val_preds
    oof_probabilities[v_idx] = val_probs
    
    f_acc = accuracy_score(y_va, val_preds)
    f_prec = precision_score(y_va, val_preds, zero_division=0)
    f_rec = recall_score(y_va, val_preds, zero_division=0)
    f_f1 = f1_score(y_va, val_preds, average="macro")
    f_auc = roc_auc_score(y_va, val_probs)
    
    fold_evaluations.append({
        "fold": f_num,
        "accuracy": float(f_acc),
        "precision": float(f_prec),
        "recall": float(f_rec),
        "macro_f1": float(f_f1),
        "roc_auc": float(f_auc)
    })
    print(f"Fold {f_num} -> Acc: {f_acc:.4f}, Prec: {f_prec:.4f}, Rec: {f_rec:.4f}, Macro-F1: {f_f1:.4f}, AUC: {f_auc:.4f}")

total_duration_ms = (time.time() - start_time) * 1000
mean_metrics = {
    "accuracy": float(np.mean([f["accuracy"] for f in fold_evaluations])),
    "precision": float(np.mean([f["precision"] for f in fold_evaluations])),
    "recall": float(np.mean([f["recall"] for f in fold_evaluations])),
    "macro_f1": float(np.mean([f["macro_f1"] for f in fold_evaluations])),
    "roc_auc": float(np.mean([f["roc_auc"] for f in fold_evaluations])),
    "f1_std": float(np.std([f["macro_f1"] for f in fold_evaluations]))
}
print(f"\nCross-Validation Summary (XGBoost):\n{json.dumps(mean_metrics, indent=2)}")


### 5. Export XGBoost Metrics and Feature Importances

In [ ]:
full_xgb = XGBClassifier(
    n_estimators=best_params["n_estimators"],
    max_depth=best_params["max_depth"],
    learning_rate=best_params["learning_rate"],
    subsample=best_params["subsample"],
    eval_metric="logloss",
    random_state=RANDOM_SEED,
    n_jobs=-1
)
full_xgb.fit(X_train_full, y_train_full)

df_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": full_xgb.feature_importances_
}).sort_values(by="importance", ascending=False).reset_index(drop=True)
df_importance.to_csv(OUTPUT_DIR / "xgboost_feature_importance.csv", index=False)

metrics_payload = {
    "model_name": "XGBoost",
    "model_type": "gradient_boosting_extreme",
    "best_hyperparameters": best_params,
    "cv_metrics_mean": mean_metrics,
    "fold_details": fold_evaluations,
    "total_cv_time_ms": round(total_duration_ms, 2),
    "top_5_features": df_importance.head(5).to_dict(orient="records")
}

with open(OUTPUT_DIR / "xgboost_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)
print(f"XGBoost metrics exported successfully to: {OUTPUT_DIR / 'xgboost_metrics.json'}")


### Findings & Observations

- **Extreme Gradient Boosting Performance:** Evaluated second-order gradient tree boosting across 5 folds.
- **Gain-Based Feature Distribution:** Observed non-linear feature splits on ratio and socioeconomic features.
- **Quarantine Adherence:** Final test set quarantined; all metrics computed strictly out-of-fold.

### Completion Checklist

- [ ] Stratified 5-Fold CV executed on training development set.
- [ ] Learning rate and depth tuned on out-of-fold Macro F1.
- [ ] Out-of-fold metrics exported to `xgboost_metrics.json`.
- [ ] Gain-based feature importances saved to `xgboost_feature_importance.csv`.
- [ ] Zero test set leakage confirmed.